### import

In [1]:
import sys
sys.path.append('/home/wuct/ALICE/reps/hf-vn-dev/dev-v0/utils/')
from utils_thn import GetTHnInfo
charm_info = GetTHnInfo.thn('charm_bulk')
bulk_info = GetTHnInfo.thn('bulk')
print(charm_info.axis_name_id_map)
print(bulk_info.axis_name_id_map)

{'Mass': 0, 'Cent': 1, 'Pt': 2, 'Sign': 3, 'ScoreBkg': 4, 'ScoreFD': 5, 'Eta': 6, 'MeanPtA': 7, 'MeanPtB': 8, 'PtProduct': 9}
{'Cent': 0, 'MeanPtA': 1, 'MeanPtB': 2, 'MeanPtProduct': 3, 'NTracksA': 4, 'NTracksB': 5}


### config

In [2]:
file_path = '/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/AnalysisResults.root'
pt_bins = [0.2, 1, 2, 3, 4, 5, 6]


#### THn QA

In [3]:
import ROOT
colors = [
    # ROOT.kBlack,       
    ROOT.kRed-4,       
    ROOT.kBlue-4, 
    ROOT.kGreen+2,     
    ROOT.kOrange+7,   
    # ROOT.kYellow-7,    
    ROOT.kMagenta-3,   
    ROOT.kCyan-3,      
    ROOT.kSpring-5,    
    ROOT.kViolet-4,    
    ROOT.kTeal-5,    
    ROOT.kGray+1    
]
file = ROOT.TFile(file_path, 'READ')
thn_charm = file.Get(charm_info.thurl)
thn_bulk = file.Get(bulk_info.thurl)

pt_mins = pt_bins[:-1]
pt_maxs = pt_bins[1:]

temp_thn_charm = thn_charm.Clone('temp_thn_charm')
temp_thn_charm.GetAxis(charm_info.axis_id('eta')).SetRangeUser(-0.8, -0.2)
temp_thn_charm.GetAxis(charm_info.axis_id('scorebkg')).SetRangeUser(0, 0.02)
a_side_charm = temp_thn_charm.Clone('a_side_charm')
temp_thn_bulk = thn_bulk.Clone('temp_thn_bulk')
a_side_bulk = temp_thn_bulk.Clone('a_side_bulk')

temp_thn_charm = thn_charm.Clone('temp_thn_charm')
temp_thn_charm.GetAxis(charm_info.axis_id('eta')).SetRangeUser(0.2, 0.8)
temp_thn_charm.GetAxis(charm_info.axis_id('scorebkg')).SetRangeUser(0, 0.02)
b_side_charm = temp_thn_charm.Clone('b_side_charm')
temp_thn_bulk = thn_bulk.Clone('temp_thn_bulk')
b_side_bulk = temp_thn_bulk.Clone('b_side_bulk')

hMeanPt_bs = []
hMeanPt_as = []
hNum_bs = []
hNum_as = []
hPtProduct_bs = []
hPtProduct_as = []
meanpt_bins = [0.0, 0.71, 0.73, 0.75, 0.77, 0.8, 1.49]
for i_meanpt, (meanpt_min, meanpt_max) in enumerate(zip(meanpt_bins[:-1], meanpt_bins[1:])):
    ## track number of tracks for each pt of charm
    a_side_bulk.GetAxis(bulk_info.axis_id('mean_pt_b')).SetRangeUser(meanpt_min*1.0001, meanpt_max*0.9999)
    temp = a_side_bulk.Projection(bulk_info.axis_id('NTracksB'))
    temp.SetName(f'NTracksB_{i_meanpt}')
    hNum_as.append(temp) # opposite side for number of tracks
        ## track number of tracks for each pt of charm
    b_side_bulk.GetAxis(bulk_info.axis_id('mean_pt_a')).SetRangeUser(meanpt_min*1.0001, meanpt_max*0.9999)
    temp = b_side_bulk.Projection(bulk_info.axis_id('NTracksA'))
    temp.SetName(f'NTracksA_{i_meanpt}')
    hNum_bs.append(temp) # opposite side for number of tracks
for i_pt, (pt_min, pt_max) in enumerate(zip(pt_mins, pt_maxs)):
    print(f'pt bin {i_pt}: {pt_min} - {pt_max} GeV/c')
    # a side
    ## track mean pt for each pt of charm
    a_side_charm.GetAxis(charm_info.axis_id('pT')).SetRangeUser(pt_min, pt_max)
    temp = a_side_charm.Projection(charm_info.axis_id('mean_pt_b'))
    temp.SetName(f'mean_pt_b_{i_pt}')
    hMeanPt_as.append(temp) # opposite side for mean pt

    
    ## pt product for each pt of charm
    temp = a_side_charm.Projection(charm_info.axis_id('pt_product'))
    temp.SetName(f'pt_product_{i_pt}')
    hPtProduct_as.append(temp) # slice by eta for pt product
    
    # b side
    ## track mean pt for each pt of charm
    b_side_charm.GetAxis(charm_info.axis_id('pT')).SetRangeUser(pt_min, pt_max)
    temp = b_side_charm.Projection(charm_info.axis_id('mean_pt_a'))
    temp.SetName(f'mean_pt_a_{i_pt}')
    hMeanPt_bs.append(temp) # opposite side for mean pt
    

    
    ## pt product for each pt of charm
    temp = b_side_charm.Projection(charm_info.axis_id('pt_product'))
    temp.SetName(f'pt_product_{i_pt}')
    hPtProduct_bs.append(temp) # slice by eta for pt product

output = ROOT.TFile('charm_bulk_results.root', 'RECREATE')

# canvas for mean pt in different charm pt bins
# a side
a_max_y_mean_pt = max(h.GetMaximum() for h in hMeanPt_as)
cMeanPt_as = ROOT.TCanvas('cMeanPt_as', 'cMeanPt_as', 1600, 1200)
leg_as = ROOT.TLegend(0.6, 0.6, 0.9, 0.9)
leg_as.SetHeader('a side charm: -0.8 < #eta < -0.2')
leg_as.SetTextSize(0.03)
meanLines = []
for i_pt, hMeanPt_a in enumerate(hMeanPt_as):
    hMeanPt_a.SetLineColor(colors[i_pt])
    hMeanPt_a.SetLineWidth(3)
    hMeanPt_a.SetMarkerColor(colors[i_pt])
    hMeanPt_a.SetMarkerSize(4)
    meanLine = ROOT.TLine(hMeanPt_a.GetMean(), 0, hMeanPt_a.GetMean(), a_max_y_mean_pt)
    meanLines.append(meanLine)
    meanLine.SetLineColor(colors[i_pt])
    meanLine.SetLineStyle(ROOT.kDashed)
    meanLine.SetLineWidth(3)
    # hMeanPt_a.SetTitle(f'Mean track pT in different charm pT bins (a side charm)')
    if i_pt == 0:
        hMeanPt_a.Draw('l')
        hMeanPt_a.GetYaxis().SetRangeUser(0, a_max_y_mean_pt * 1.2)
    else:
        hMeanPt_a.Draw('l same')
    meanLine.Draw('same')

    leg_as.AddEntry(hMeanPt_a, f'{pt_mins[i_pt]} < pT < {pt_maxs[i_pt]} GeV/c', 'lp')
# cMeanPt_as.BuildLegend()
leg_as.Draw()
output.cd()
cMeanPt_as.Write()
cMeanPt_as.SaveAs('cMeanPt_as.png')

# b side
b_max_y_mean_pt = max(h.GetMaximum() for h in hMeanPt_bs)
cMeanPt_bs = ROOT.TCanvas('cMeanPt_bs', 'cMeanPt_bs', 1600, 1200)
leg_bs = ROOT.TLegend(0.6, 0.6, 0.9, 0.9)
leg_bs.SetHeader('b side charm: 0.2 < #eta < 0.8')
leg_bs.SetTextSize(0.03)
meanLines = []
for i_pt, hMeanPt_b in enumerate(hMeanPt_bs):
    hMeanPt_b.SetLineColor(colors[i_pt])
    hMeanPt_b.SetMarkerColor(colors[i_pt])
    hMeanPt_b.SetLineWidth(3)
    hMeanPt_b.SetMarkerSize(4)
    meanLines.append(ROOT.TLine(hMeanPt_b.GetMean(), 0, hMeanPt_b.GetMean(), b_max_y_mean_pt))
    meanLine = meanLines[-1]
    meanLine.SetLineColor(colors[i_pt])
    meanLine.SetLineStyle(ROOT.kDashed)
    meanLine.SetLineWidth(3)
    # hMeanPt_b.SetTitle(f'Mean track pT in different charm pT bins (b side charm)')
    if i_pt == 0:
        hMeanPt_b.Draw('l')
        hMeanPt_b.GetYaxis().SetRangeUser(0, b_max_y_mean_pt * 1.2)
    else:
        hMeanPt_b.Draw('l same')
    meanLine.Draw('same')
    leg_bs.AddEntry(hMeanPt_b, f'{pt_mins[i_pt]} < pT < {pt_maxs[i_pt]} GeV/c', 'lp')
# cMeanPt_bs.BuildLegend()
leg_bs.Draw()
output.cd()
cMeanPt_bs.Write()
cMeanPt_bs.SaveAs('cMeanPt_bs.png')
output.Close()

# canvas for number of tracks in different charm pt bins
# a side
cNum_as = ROOT.TCanvas('cNum_as', 'cNum_as', 1600, 1200)

leg_num_as = ROOT.TLegend(0.6, 0.6, 0.9, 0.9)
leg_num_as.SetHeader('a side charm: -0.8 < #eta < -0.2')
leg_num_as.SetTextSize(0.03)
for i_pt, hNum_a in enumerate(hNum_as):
    if i_pt == 0:
        hNum_a.GetXaxis().SetRangeUser(0, 400)
    hNum_a.SetLineColor(i_pt + 1)
    hNum_a.SetLineWidth(2)
    hNum_a.SetMarkerSize(4)
    # hNum_a.SetTitle(f'Number of tracks in different charm pT bins (a side charm)')
    hNum_a.Draw('same')
    leg_num_as.AddEntry(hNum_a, f'{meanpt_bins[i_pt]} < pT < {meanpt_bins[i_pt + 1]} GeV/c', 'lp')
# cNum_as.BuildLegend()
leg_num_as.Draw()
cNum_as.SaveAs('cNum_as.png')
# b side
cNum_bs = ROOT.TCanvas('cNum_bs', 'cNum_bs', 1600, 1200)
leg_num_bs = ROOT.TLegend(0.6, 0.6, 0.9, 0.9)
leg_num_bs.SetHeader('b side charm: 0.2 < #eta < 0.8')
leg_num_bs.SetTextSize(0.03)
for i_pt, hNum_b in enumerate(hNum_bs):
    if i_pt == 0:
        hNum_b.GetXaxis().SetRangeUser(0, 400)
    hNum_b.SetLineColor(i_pt + 1)
    hNum_b.SetLineWidth(2)
    hNum_b.SetMarkerSize(4)
    # hNum_b.SetTitle(f'Number of tracks in different charm pT bins (b side charm)')
    hNum_b.Draw('same')
    leg_num_bs.AddEntry(hNum_b, f'{meanpt_bins[i_pt]} < pT < {meanpt_bins[i_pt + 1]} GeV/c', 'lp')
# cNum_bs.BuildLegend()
leg_num_bs.Draw()
cNum_bs.SaveAs('cNum_bs.png')

# canvas for pt product in different charm pt bins
# a side
a_max_y_pt_product = max(h.GetMaximum() for h in hPtProduct_as)
a_min_y_pt_product = min(h.GetMinimum() for h in hPtProduct_as)
cPtProduct_as = ROOT.TCanvas('cPtProduct_as', 'cPtProduct_as', 1600, 1200)
leg_pt_product_as = ROOT.TLegend(0.6, 0.6, 0.9, 0.9)
leg_pt_product_as.SetHeader('a side charm: -0.8 < #eta < -0.2')
leg_pt_product_as.SetTextSize(0.03)
for i_pt, hPtProduct_a in enumerate(hPtProduct_as):
    if i_pt == 0:
        hPtProduct_a.GetXaxis().SetRangeUser(0, 10)
        hPtProduct_a.GetYaxis().SetRangeUser(0, a_max_y_pt_product * 1.2)
    hPtProduct_a.SetLineColor(colors[i_pt])
    hPtProduct_a.SetLineWidth(3)
    hPtProduct_a.SetMarkerColor(colors[i_pt])
    hPtProduct_a.SetMarkerSize(4)
    # hPtProduct_a.SetTitle(f'Pt product in different charm pT bins (a side charm)')
    if i_pt == 0:
        hPtProduct_a.Draw('l')
    else:
        hPtProduct_a.Draw('l same')
    leg_pt_product_as.AddEntry(hPtProduct_a, f'{pt_mins[i_pt]} < pT < {pt_maxs[i_pt]} GeV/c', 'lp')
#cPtProduct_as.BuildLegend()
leg_pt_product_as.Draw()
cPtProduct_as.SaveAs('cPtProduct_as.png')
# b side
b_max_y_pt_product = max(h.GetMaximum() for h in hPtProduct_bs)
b_min_y_pt_product = min(h.GetMinimum() for h in hPtProduct_bs)
cPtProduct_bs = ROOT.TCanvas('cPtProduct_bs', 'cPtProduct_bs', 1600, 1200)
leg_pt_product_bs = ROOT.TLegend(0.6, 0.6, 0.9, 0.9)
leg_pt_product_bs.SetHeader('b side charm: 0.2 < #eta < 0.8')
leg_pt_product_bs.SetTextSize(0.03)
for i_pt, hPtProduct_b in enumerate(hPtProduct_bs):
    if i_pt == 0:
        hPtProduct_b.GetXaxis().SetRangeUser(0, 10)
        hPtProduct_b.GetYaxis().SetRangeUser(0, b_max_y_pt_product * 1.2)
    hPtProduct_b.SetLineColor(colors[i_pt])
    hPtProduct_b.SetLineWidth(3)
    hPtProduct_b.SetMarkerColor(colors[i_pt])
    hPtProduct_b.SetMarkerSize(4)
    # hPtProduct_b.SetTitle(f'Pt product in different charm pT bins (b side charm)')
    if i_pt == 0:
        hPtProduct_b.Draw('l')
    else:
        hPtProduct_b.Draw('l same')
    leg_pt_product_bs.AddEntry(hPtProduct_b, f'{pt_mins[i_pt]} < pT < {pt_maxs[i_pt]} GeV/c', 'lp')
# cPtProduct_bs.BuildLegend()
leg_pt_product_bs.Draw()
cPtProduct_bs.SaveAs('cPtProduct_bs.png')


pt bin 0: 0.2 - 1 GeV/c
pt bin 1: 1 - 2 GeV/c
pt bin 2: 2 - 3 GeV/c
pt bin 3: 3 - 4 GeV/c
pt bin 4: 4 - 5 GeV/c
pt bin 5: 5 - 6 GeV/c


Info in <TCanvas::Print>: png file cMeanPt_as.png has been created
Info in <TCanvas::Print>: png file cMeanPt_bs.png has been created
Info in <TCanvas::Print>: png file cNum_as.png has been created
Info in <TCanvas::Print>: png file cNum_bs.png has been created
Info in <TCanvas::Print>: png file cPtProduct_as.png has been created
Info in <TCanvas::Print>: png file cPtProduct_bs.png has been created


In [6]:

import ROOT
file_path = '/home/wuct/ALICE/reps/cfAnRes/tools/notebook/charmbulk/charm_bulk_results.root'
file = ROOT.TFile(file_path, 'READ')
cMeanPt_bs = file.Get('cMeanPt_bs')
cMeanPt_as = file.Get('cMeanPt_as')

def get_hists(canvas):
    return [obj for obj in canvas.GetListOfPrimitives() if obj.InheritsFrom("TH1")]

hists_bs = get_hists(cMeanPt_bs)
hists_as = get_hists(cMeanPt_as)

def find_equal_integral_bins(hist, n_div=6):
    total = hist.Integral()
    target = total / n_div
    bins = []
    total = 0
    for i in range(1, hist.GetNbinsX() + 1):
        total += hist.GetBinContent(i)
        if total >= target or i == hist.GetNbinsX() and len(bins) < n_div:
            print(f"Bin {i} has cumulative integral {total:.2f}, target was {target:.2f}")
            bins.append(i-1)
            print(f"Added bin edge at bin {i-1} with x value {hist.GetBinCenter(i-1):.3f}")
            print(f"The real cumulative integral shouold be {(total - hist.GetBinContent(i)):.2f}")
            total -= target
            print(f"Reset total to {total:.2f} after subtracting target")
            print("----\n")
    return bins

n_div = 5
bin_edges_bs = {}
bin_edges_as = {}
for h in hists_bs:
    bin_edges_bs[h.GetName()] = find_equal_integral_bins(h, n_div)

for h in hists_as:
    bin_edges_as[h.GetName()] = find_equal_integral_bins(h, n_div)

def get_x_positions(hists, bin_edges):
    result = []
    for h in hists:
        result.append([h.GetBinCenter(b) for b in bin_edges[h.GetName()]])
    return result

x_positions_bs = get_x_positions(hists_bs, bin_edges_bs)
x_positions_as = get_x_positions(hists_as, bin_edges_as)
for h, xs in zip(hists_bs, x_positions_bs):
    print(f"{h.GetName()}: {[f'{x:.3f}' for x in xs]}")

def draw_lines_with_ranges(canvas, hists, x_positions, n_div):
    canvas.cd()
    kept = []
    global_max = max(h.GetMaximum() for h in hists)
    for i_h, h in enumerate(hists):
        if i_h == 0:
            h.GetYaxis().SetRangeUser(0, global_max * 1.2)
        h.Draw("l same")
    canvas.Update()
    ymax = canvas.GetUymax()
    xmin_axis = canvas.GetUxmin()
    xmax_axis = canvas.GetUxmax()
    label_y = global_max / 2.0
    for h, xs in zip(hists, x_positions):
        ytop = h.GetMaximum()
        color = h.GetLineColor()
        for x in xs:
            line = ROOT.TLine(x, 0, x, ytop)
            line.SetLineWidth(2)
            line.SetLineColor(color)
            line.SetLineStyle(ROOT.kDashed)
            line.Draw()
            kept.append(line)
    for i in range(n_div):
        xs_at_i = [xs[i] for xs in x_positions if i < len(xs)]
        if len(xs_at_i) < 2:
            continue
        x_min = min(xs_at_i)
        x_max = max(xs_at_i)
        label_x = xmin_axis*1.2 + (i + 0.5) * (xmax_axis - xmin_axis) / n_div
        label = ROOT.TLatex()
        label.SetTextSize(0.025)
        label.SetTextAlign(21)
        label.SetNDC(False)
        label.DrawLatex(label_x, label_y,
                        f"[{x_min:.3f}]")
        kept.append(label)
    canvas.Modified()
    canvas.Update()
    return kept
canvas_bs = ROOT.TCanvas('cMeanPt_bs', 'cMeanPt_bs', 1600, 1200)
kept_bs = draw_lines_with_ranges(canvas_bs, hists_bs, x_positions_bs, n_div)
canvas_bs.SaveAs('cMeanPt_bs_with_ranges.png')

canvas_as = ROOT.TCanvas('cMeanPt_as', 'cMeanPt_as', 1600, 1200)
kept_as = draw_lines_with_ranges(canvas_as, hists_as, x_positions_as, n_div)
canvas_as.SaveAs('cMeanPt_as_with_ranges.png')

output = ROOT.TFile('charm_bulk_results_with_ranges.root', 'RECREATE')
canvas_bs.Write()
canvas_as.Write()
output.Close()


Bin 72 has cumulative integral 18524.00, target was 16212.60
Added bin edge at bin 71 with x value 0.705
The real cumulative integral shouold be 13378.00
Reset total to 2311.40 after subtracting target
----

Bin 75 has cumulative integral 21712.40, target was 16212.60
Added bin edge at bin 74 with x value 0.735
The real cumulative integral shouold be 14871.40
Reset total to 5499.80 after subtracting target
----

Bin 77 has cumulative integral 18641.80, target was 16212.60
Added bin edge at bin 76 with x value 0.755
The real cumulative integral shouold be 12271.80
Reset total to 2429.20 after subtracting target
----

Bin 80 has cumulative integral 18574.20, target was 16212.60
Added bin edge at bin 79 with x value 0.785
The real cumulative integral shouold be 14032.20
Reset total to 2361.60 after subtracting target
----

Bin 98 has cumulative integral 16212.60, target was 16212.60
Added bin edge at bin 97 with x value 0.965
The real cumulative integral shouold be 16211.60
Reset total to

Warning in <TCanvas::Constructor>: Deleting canvas with same name: cMeanPt_bs
Info in <TCanvas::Print>: png file cMeanPt_bs_with_ranges.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: cMeanPt_as
Info in <TCanvas::Print>: png file cMeanPt_as_with_ranges.png has been created
